In [65]:
from __future__ import annotations
# ═════════════ IMPORTS & CONFIG ═════════════════════════════════════════
import logging, warnings, joblib
from pathlib import Path
from typing   import Dict, List, Tuple
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold, chi2, f_classif
from sklearn.preprocessing      import OneHotEncoder, MinMaxScaler
from sklearn.metrics            import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from keras.layers import LSTM, GRU, Dense, Dropout, Bidirectional
from keras.regularizers import l2

from statsmodels.tsa.seasonal import seasonal_decompose

# warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
tf.config.run_functions_eagerly(False)
tf.keras.utils.set_random_seed(42)

# Optionally limit CPU threads (or set to all with os.cpu_count())
tf.config.threading.set_intra_op_parallelism_threads(os.cpu_count())
tf.config.threading.set_inter_op_parallelism_threads(os.cpu_count())

# ═════════════ RUTAS & CONSTANTES ═══════════════════════════════════════
BASE_PATH      = Path("../")
DATA_BASE_PATH = BASE_PATH / "data/pipeline"
PREPARED_BASE_PATH   = DATA_BASE_PATH / "prepared"
RESULTS_PATH   = BASE_PATH / "resultados/iteracion_6"

DATA_FROM = {
    "train": PREPARED_BASE_PATH / "train.csv",
    "test": PREPARED_BASE_PATH / "test.csv",
    "submission": PREPARED_BASE_PATH / "sample_submission.csv"
}

LOW_VAR, HIGH_CORR = 0.01, 0.95

# ═════════════ ARCHIVOS ═══════════════════════════════════════
import sys
sys.path.append('../pipeline')
from preprocess import feature_engineering



In [66]:
def get_df(file_name: str, teq_type: str = "train") -> pd.DataFrame:
    """
    Carga los DataFrames de entrenamiento y prueba desde las rutas especificadas.
    """
    train_df = pd.read_csv(PREPARED_BASE_PATH / f"{file_name}_{teq_type}_train.csv")
    test_df = pd.read_csv(PREPARED_BASE_PATH / f"{file_name}_{teq_type}_test.csv")
    
    return pd.concat([train_df, test_df])
    
    

# LSTM Auxiliary function

In [67]:
## Auxiliares LSTM
def sliding_window_lstm(Xdata_df: pd.DataFrame, ydata_df: pd.Series, window_size: int, optional_cols: List[str] = None):
    X, y = [], []
    dict_col_array = {}

    #Create a sliding window
    for i in range(window_size, len(ydata_df)):
        X.append(ydata_df.iloc[i-window_size:i].values) # Agregamos ydata para entrenar y predecir un paso en el futuro
        y.append(ydata_df.iloc[i])

    if optional_cols is not None and len(optional_cols) > 0:
        for col in optional_cols:
            aux_arr = []
            for i in range(window_size, len(Xdata_df)):
                aux_arr.append(Xdata_df.iloc[i-window_size:i][col].values)
                
            aux_arr = np.array(aux_arr)
            dict_col_array[col] = aux_arr
            

    X, y = np.array(X), np.array(y)

    if optional_cols is not None and len(optional_cols) > 0:
        X = X.reshape(X.shape[0], X.shape[1], 1)

        for _, val in dict_col_array.items():
            X = np.insert(X, -1, val, axis=2)
    else:
        X = np.reshape(X, (X.shape[0], X.shape[1], 1 ))

    return X, y

def get_seasonal_decompose_data(data_df: pd.DataFrame, target_col: str = "Valor"):
    res_decompose = seasonal_decompose(data_df[target_col], model='additive', extrapolate_trend='freq')

    data_df["seasonal"] = res_decompose.seasonal
    data_df["trend"] = res_decompose.trend
    data_df["resid"] = res_decompose.resid

    return data_df

def plot_decompose_data(data_decomposed_df: pd.DataFrame, observed_col: str = "Valor", v_line: float = 0):
    # res_decompose = seasonal_decompose(full_sc["Valor"], model='additive', extrapolate_trend='freq')

    fig, axs = plt.subplots(nrows=4, ncols=1, figsize=(18, 12), sharex=True)
    data_decomposed_df[observed_col].plot(ax=axs[0])
    axs[0].set_title('Serie original', fontsize=12)
    axs[0].grid()

    data_decomposed_df["trend"].plot(ax=axs[1])
    axs[1].set_title('Tendencia', fontsize=12)
    axs[1].grid()

    data_decomposed_df["seasonal"].plot(ax=axs[2])
    axs[2].set_title('Estacionalidad', fontsize=12)
    axs[2].grid()

    data_decomposed_df["resid"].plot(ax=axs[3])
    axs[3].set_title('Residuos', fontsize=12)
    axs[3].grid()

    if v_line > 0:
        for i in range(len(axs)):
            xmin, xmax = axs[i].get_xlim()
            x_position = xmin + (xmax - xmin) * 0.8
            axs[i].axvline(x=x_position, color='r', linestyle='--')


    fig.suptitle('Descomposición de la serie original vs serie diferenciada', fontsize=14)
    fig.tight_layout()

def lstm_model(input_shape):
    # Build the Model
    model = keras.models.Sequential()

    model.add(keras.layers.LSTM(64, return_sequences=True, input_shape=input_shape))
    model.add(keras.layers.LSTM(64, return_sequences=False))
    model.add(keras.layers.Dense(128, activation="relu"))
    model.add(keras.layers.Dense(1))

    # model.summary()
    model.compile(optimizer="adam",
                loss="mae",
                metrics=[keras.metrics.RootMeanSquaredError()])
    
    return model

def Xy_split(train_df: pd.DataFrame, target_col: str = "Valor"):
    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col]

    return X_train, y_train



In [68]:
def walk_forward_validation(data_df: pd.DataFrame, target_col:str = "Valor", training_window: int = 36, test_window: int = 1):
    """
    Walk-forward validation for time series forecasting.
    """
    actuals = []
    predictions = []
    prediction_seas_trends = []
    prediction_resids = []
    metrics_trend_season = []
    sliding_window_size = 24

    for i in range(0,
                   len(data_df) - training_window - test_window,
                    test_window):
        
        # Split into train-test sets
        train_data = data_df[i:i + training_window]
        test_data = data_df[i + training_window : i + training_window + test_window]

        train_data, ohe, minmax_sc = feature_engineering(train_data, "", target_col, LOW_VAR, HIGH_CORR, ignore_lags=True)
        test_data, _, _ = feature_engineering(test_data, "", target_col, LOW_VAR, HIGH_CORR, ohe, minmax_sc, train_data.columns.to_list(), ignore_lags=True)

        train_data = train_data.set_index('Fecha')
        test_data = test_data.set_index('Fecha')

        # Decompose
        train_dec_df = get_seasonal_decompose_data(train_data, target_col)
        
        # ════════════════════════════════════════════════════ Prediccion con Trend y Seasonal ═══════════════════════════════════════
        # Divide into X and y
        X_train, y_train,  = Xy_split(train_dec_df, target_col)
        X_test, y_test = Xy_split(test_data, target_col)    

        # Create sliding window
        X_train_sw, y_train_sw = sliding_window_lstm(X_train, y_train, window_size=sliding_window_size, optional_cols=["seasonal", "trend"])

        # Create the model and fit for the data using seasonal and trend data
        model = lstm_model((X_train_sw.shape[1], X_train_sw.shape[2]))
        model.fit(X_train_sw, y_train_sw, epochs=100, batch_size=32, verbose=0)

        # Make a Prediction
        ltsm_predictions_seas_trend = model.predict(X_train_sw[-1:].copy())
        # Calculate metrics
        mae_sc=mean_absolute_error(y_test, ltsm_predictions_seas_trend)
        rmse_sc=mean_squared_error(y_test, ltsm_predictions_seas_trend)**0.5
        r2_sc  =r2_score(y_test,ltsm_predictions_seas_trend)
        metrics_trend_season.append({
            "MAE": mae_sc,
            "RMSE": rmse_sc,
            "R2": r2_sc
        })
        
        unscaled_prediction = minmax_sc.inverse_transform(np.pad(ltsm_predictions_seas_trend, ((0, 0), (0, 5)), mode='constant'))
        # ════════════════════════════════════════════════════ Prediccion del residuo (ruido) ═══════════════════════════════════════
        X_train, y_train,  = Xy_split(train_dec_df, "resid")

        # Create sliding window
        X_train_sw, y_train_sw = sliding_window_lstm(X_train, y_train, window_size=sliding_window_size, optional_cols=['Mes_Aug', 'Mes_Dec', 'Mes_Feb', 'Mes_Jan', 'Mes_Jul', 'Mes_Jun', 'Mes_Mar', 'Mes_May', 'Mes_Nov', 'Mes_Oct', 'Mes_Sep'])

        # Create the model and fit for the data using seasonal and trend data
        model_resid = lstm_model((X_train_sw.shape[1], X_train_sw.shape[2]))
        model_resid.fit(X_train_sw, y_train_sw, epochs=100, batch_size=32, verbose=0)

        # Make a Prediction
        ltsm_predictions_resid = model_resid.predict(X_train_sw[-1:].copy())
        # Calculate metrics # NOTE: These metrics cannot be done during this process as there's no reliable way of knowing the next residuals
        # mae_sc=mean_absolute_error(y_test, ltsm_predictions_seas_trend)
        # rmse_sc=mean_squared_error(y_test, ltsm_predictions_seas_trend)**0.5
        # r2_sc  =r2_score(y_test,ltsm_predictions_seas_trend)
        # metrics_trend_season.append({
        #     "MAE": mae_sc,
        #     "RMSE": rmse_sc,
        #     "R2": r2_sc
        # })
        
        unscaled_prediction_resid = minmax_sc.inverse_transform(np.pad(ltsm_predictions_resid, ((0, 0), (0, 5)), mode='constant'))
        
        prediction_seas_trend = unscaled_prediction[0, 0:1]
        prediction_resid = unscaled_prediction_resid[0, 0:1]
        prediction = unscaled_prediction[0, 0:1] + unscaled_prediction_resid[0, 0:1]

        predictions.extend(prediction)
        prediction_seas_trends.extend(prediction_seas_trend)
        prediction_resids.extend(prediction_resid)
        actuals.extend(y_test.values)

        

    # return predictions, actuals, metrics_trend_season, metrics_resid
    return {
        "predictions": {
            "seas_trend": prediction_seas_trends,
            "resid": prediction_resids,
            "total": predictions
        },
        "actuals": actuals,
        "metrics_trend_season": metrics_trend_season
    }

In [69]:
full_sc = get_df("exportaciones_pais", "teq")
full_sc

,Valor,Year,Mes
0,5829070.98,1997,1
1,5532315.75,1997,2
2,5265032.53,1997,3
3,5745381.50,1997,4
4,5561477.54,1997,5
...,...,...,...
63,4780939.88,2024,11
64,6789662.05,2024,12
65,6444358.59,2025,1
66,4990450.92,2025,2


In [70]:
resultados = walk_forward_validation(full_sc, target_col="Valor", training_window=36, test_window=1)

INFO | ··· FE:  ···
INFO | ··· FE:  ···


Creating new OneHotEncoder
Creating new MinMaxScaler
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step


INFO | ··· FE:  ···
INFO | ··· FE:  ···


Creating new OneHotEncoder
Creating new MinMaxScaler
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 347ms/step


INFO | ··· FE:  ···
INFO | ··· FE:  ···


...

Creating new OneHotEncoder
Creating new MinMaxScaler
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 546ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 590ms/step


INFO | ··· FE:  ···
INFO | ··· FE:  ···


Creating new OneHotEncoder
Creating new MinMaxScaler
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step


INFO | ··· FE:  ···
INFO | ··· FE:  ···


Creating new OneHotEncoder
Creating new MinMaxScaler
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 421ms/step


In [77]:
resultados["predictions"]["total"]

array([12062058.], dtype=float32)

In [75]:
len(resultados["predictions"]["seas_trend"])
# len(full_sc.index) - 

1

In [76]:
wf_df = full_sc[37:].copy()
wf_df["Fecha"] = pd.to_datetime(
        wf_df["Year"].astype(str) + "-" + wf_df["Mes"].astype(str).str.zfill(2)
    )
wf_df = wf_df.set_index('Fecha')


# wf_df["Walk_forward_seas_trend"] = resultados["predictions"]["seas_trend"]
wf_df["Walk_forward_total"] = resultados["predictions"]["total"]

plt.figure(figsize=(12,8))
plt.plot(wf_df.index, wf_df['Valor'], label="Actual Values")
plt.plot(wf_df.index, wf_df["Walk_forward_seas_trend"], label="Walk forward Predictions LSTM - Season/trend", linestyle='--')
plt.plot(wf_df.index, wf_df["Walk_forward_total"], label="Walk forward Predictions LSTM - Season/trend + Residuals", linestyle='--')

plt.title("LSTM Walk forward strategy vs Actual Values")
plt.xlabel("Date")
plt.ylabel("Exportacion")
plt.legend()
plt.show()

ValueError: Length of values (1) does not match length of index (302)